****

**Nombre:** Juan Martín Cusme Carrera

**Materia:** Recuperación de información

**Curso:** GR1CC

**Fecha:** 18/11/2026

****

# Examen Recuperación de Información con SciFact

Se desarrolla un sistema de búsqueda sobre el corpus científico **SciFact** para comparar dos formas de recuperar documentos: `TF-IDF` con similitud coseno y `BM25`. A partir de una consulta escrita por el usuario, ambos modelos calculan sus propias puntuaciones y generan rankings independientes. El objetivo es comprobar qué documentos recupera cada modelo y qué tan bien coinciden con los documentos relevantes anotados en el corpus.

En este notebook se presenta el código completo, se ejecutan los bloques y se conservan sus resultados. Primero se prepara el texto y se construye el índice; después se evalúan ambos modelos con las consultas de `test` y se interpretan las diferencias observadas. Finalmente, se prueba el servicio publicado en **https://exb1-martincusme-scifact.vercel.app**, que ofrece los endpoints `GET /health` y `POST /search`.

Los documentos y las consultas son de [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact), y los juicios de relevancia provienen de [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels). Para reproducir la ejecución, se instalan las dependencias con `python -m pip install -r requirements.txt` y se ejecutan los bloques en orden.

# Fase 1: Preprocesamiento y definición de los modelos

Se define la función `preprocess`, que aplica el mismo tratamiento a los documentos y a las consultas. El texto se normaliza con `NFKD`, se convierte a minúsculas y se eliminan los diacríticos. Luego se separa en tokens alfanuméricos y se retiran las palabras incluidas en `STOPWORDS`, como artículos, pronombres y preposiciones frecuentes del inglés. Se conservan los números y las negaciones porque pueden aportar información en las afirmaciones científicas.

Después se aplica stemming en inglés con `Snowball/Porter2`, mediante `STEMMER`. Este paso reduce variantes de una palabra a una raíz común para facilitar las coincidencias entre consulta y documento. Por ejemplo, formas plurales y singulares pueden terminar representadas por el mismo término. Se utiliza stemming y no se añade lematización; se trata de una decisión explícita de preprocesamiento.

La clase `SearchEngine` define un índice invertido propio en `postings`. Para cada término se guardan los identificadores de los documentos donde aparece y su frecuencia. También se calculan las longitudes de los documentos y las estadísticas necesarias para puntuar las coincidencias. Cada documento se representa con su título y su abstract completos, conservando el valor original de `_id` como texto. No se crean fragmentos con nuevos identificadores.

En `TF-IDF`, la frecuencia del término se multiplica por un IDF suavizado: `ln((N + 1) / (df + 1)) + 1`. Aquí, `N` es el número de documentos y `df` indica en cuántos aparece el término. Se calcula la similitud coseno entre los vectores de la consulta y del documento, normalizados con sus normas L2. Así se compara la distribución de los términos ponderados y no únicamente la cantidad de palabras compartidas.

En `BM25` se combina el IDF `ln(1 + (N - df + 0.5) / (df + 0.5))` con la frecuencia del término y la longitud del documento. La frecuencia tiene un efecto de saturación: repetir una palabra muchas veces no aumenta la puntuación de forma lineal. Se fijan `k1 = 1.2` y `b = 0.75` antes de evaluar `test`; cada término distinto de la consulta aporta una sola vez a la suma. Ambos modelos se implementan con sus fórmulas, sin utilizar modelos preentrenados.

La función `rank` ordena las puntuaciones de mayor a menor y resuelve los empates por identificador ascendente. Los términos ajenos al vocabulario se ignoran y, si no hay coincidencias, se devuelve una lista vacía. En este mismo bloque se define la API que utilizará el motor; su comportamiento se comprueba en las fases siguientes. El código corresponde a la implementación de `app.py`.

El stemmer conserva estado interno durante el procesamiento. Se utiliza `STEMMER_LOCK` para proteger su acceso cuando llegan solicitudes simultáneas; esta protección mantiene la misma transformación del texto.

In [1]:
"""SciFact: modelos clasicos propios y API del examen.

Esta implementacion tambien se incluye completa en ExB1_MartínCusme_Scifact.ipynb.
"""
import gzip
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from functools import lru_cache
from threading import Lock
from pathlib import Path
from typing import Literal

import snowballstemmer
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, Field

# Lista explicita para ingles. Se conservan negaciones y numeros cientificos.
STOPWORDS = frozenset("a an the and or but if then else when where who whom which what why how of to in on at by for from with as is are was were be been being am do does did doing have has had having can could would should will may might must shall it its this that these those i me my we our you your he him his she her they them their there here such than so too very also into about between during each both other some any all only same".split())
STEMMER = snowballstemmer.stemmer("english")
STEMMER_LOCK = Lock()


def preprocess(text):
    """NFKD, minusculas, tokens alfanumericos, stopwords y Porter2 ingles."""
    text = unicodedata.normalize("NFKD", text or "").lower()
    text = "".join(c for c in text if not unicodedata.combining(c))
    tokens = re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text)
    with STEMMER_LOCK:
        return STEMMER.stemWords([t for t in tokens if t not in STOPWORDS])


class SearchEngine:
    def __init__(self, documents):
        self.documents = {}
        self.postings = defaultdict(dict)
        self.lengths = {}
        for row in documents:
            doc_id = str(row["_id"])
            if doc_id in self.documents:
                raise ValueError(f"Documento duplicado: {doc_id}")
            title, text = row.get("title") or "", row.get("text") or ""
            self.documents[doc_id] = {"title": title, "text": text}
            counts = Counter(preprocess(title + " " + text))
            self.lengths[doc_id] = sum(counts.values())
            for term, tf in counts.items():
                self.postings[term][doc_id] = tf
        self.n = len(self.documents)
        if not self.n:
            raise ValueError("Corpus vacio")
        self.avgdl = sum(self.lengths.values()) / self.n
        self.tfidf_idf = {t: math.log((self.n + 1)/(len(p) + 1)) + 1 for t, p in self.postings.items()}
        self.bm25_idf = {t: math.log1p((self.n-len(p)+0.5)/(len(p)+0.5)) for t, p in self.postings.items()}
        squared_norms = defaultdict(float)
        for term, postings in self.postings.items():
            for doc_id, tf in postings.items():
                squared_norms[doc_id] += (tf*self.tfidf_idf[term])**2
        self.norms = {d: math.sqrt(value) for d, value in squared_norms.items()}

    def rank(self, query, model="bm25", top_k=10):
        if model not in ("bm25", "tfidf"):
            raise ValueError("Modelo invalido")
        if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
            raise ValueError("top_k debe ser entero positivo")
        terms = Counter(preprocess(query))
        scores = defaultdict(float)
        if model == "tfidf":
            weights = {t: tf*self.tfidf_idf[t] for t, tf in terms.items() if t in self.tfidf_idf}
            qnorm = math.sqrt(sum(w*w for w in weights.values()))
            if not qnorm:
                return []
            for term, weight in weights.items():
                for doc_id, tf in self.postings[term].items():
                    scores[doc_id] += weight*tf*self.tfidf_idf[term]
            scores = {doc_id: score/(qnorm*self.norms[doc_id]) for doc_id, score in scores.items()}
        else:
            # Parametros fijados a priori; no se ajustan con test.
            k1, b = 1.2, 0.75
            for term in sorted(terms):
                for doc_id, tf in self.postings.get(term, {}).items():
                    norm = 1-b + b*self.lengths[doc_id]/self.avgdl
                    scores[doc_id] += self.bm25_idf[term]*tf*(k1+1)/(tf+k1*norm)
        return sorted(((d, s) for d, s in scores.items() if s > 0), key=lambda x: (-x[1], x[0]))[:top_k]

    def search(self, query, model="bm25", top_k=10):
        return [{"rank": rank, "document_id": doc_id, "score": float(score),
                 "title": self.documents[doc_id]["title"],
                 "snippet": " ".join(self.documents[doc_id]["text"].split())[:350]}
                for rank, (doc_id, score) in enumerate(self.rank(query, model, top_k), 1)]


DATA_FILE = Path(__file__).resolve().parent / ".data" / "corpus.json.gz" if "__file__" in globals() else Path.cwd()/".data"/"corpus.json.gz"


@lru_cache(maxsize=1)
def get_engine():
    if not DATA_FILE.exists():
        raise HTTPException(503, "Corpus no preparado. Ejecuta python build.py.")
    with gzip.open(DATA_FILE, "rt", encoding="utf-8") as file:
        return SearchEngine(json.load(file))


app = FastAPI(title="SciFact - Examen RI", version="1.0.0")


class SearchRequest(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")
    query: str = Field(min_length=1, max_length=10000)
    model: Literal["bm25", "tfidf"] = "bm25"
    top_k: int = Field(default=10, ge=1, strict=True)


@app.get("/health")
def health():
    engine = get_engine()
    return {"status": "ok", "corpus": "SciFact", "documents": engine.n, "models": ["tfidf", "bm25"]}


@app.post("/search")
def search(body: SearchRequest):
    return {"model": body.model, "results": get_engine().search(body.query, body.model, body.top_k)}


# Fase 2: Carga del corpus y construcción del índice

Se prepara el corpus mediante la función `prepare`. Si todavía no existe una copia local, se descarga la colección `corpus` de `BeIR/scifact` y se conservan los campos `_id`, `title` y `text`. Los documentos se guardan en `.data/corpus.json.gz` para poder reutilizarlos sin repetir la descarga en cada ejecución.

Con los documentos cargados se crea `engine`, una instancia de `SearchEngine`. En este paso se procesa la concatenación del título y del abstract, se construye el vocabulario y se llenan las listas de frecuencias del índice invertido. También se verifica que no haya identificadores duplicados, ya que los resultados deben relacionarse con los mismos documentos que aparecen en los qrels.

La salida permite comprobar que se utilizan los **5.183 documentos originales** de SciFact. Además, se muestran el tamaño del vocabulario, la longitud media después del preprocesamiento, un ejemplo de los tokens obtenidos y un identificador original. Estos datos ayudan a verificar qué información se está utilizando antes de realizar las búsquedas.

In [2]:
"""Descargar y conservar documentos completos; se ejecuta durante el build."""
import gzip
import json
from pathlib import Path
from datasets import load_dataset


def prepare():
    path = Path.cwd() / ".data" / "corpus.json.gz"
    if not path.exists():
        corpus = load_dataset("BeIR/scifact", "corpus", split="corpus")
        rows = [{"_id": str(r["_id"]), "title": r["title"], "text": r["text"]} for r in corpus]
        assert len(rows) == len({row["_id"] for row in rows})
        path.parent.mkdir(parents=True, exist_ok=True)
        with gzip.open(path, "wt", encoding="utf-8") as file:
            json.dump(rows, file, ensure_ascii=False)
    with gzip.open(path, "rt", encoding="utf-8") as file:
        rows = json.load(file)
    print(f"Corpus SciFact preparado: {len(rows)} documentos originales completos.")
    return rows



documents = prepare()
engine = SearchEngine(documents)
print("Documentos:", engine.n, "Vocabulario:", len(engine.postings), "Longitud media:", round(engine.avgdl, 2))
print("Ejemplo de preprocesamiento:", preprocess("The cells were activated in 2020, but not inhibited."))
print("Ejemplo ID original:", documents[0]["_id"])

C:\Users\DETPC\Desktop\proy_ri\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus SciFact preparado: 5183 documentos originales completos.


Documentos: 5183 Vocabulario: 28228 Longitud media: 151.85
Ejemplo de preprocesamiento: ['cell', 'activ', '2020', 'not', 'inhibit']
Ejemplo ID original: 4983


# Fase 3: Preparación de las consultas y los juicios de relevancia

Se cargan las consultas de `BeIR/scifact` y los qrels de la partición `test` de `BeIR/scifact-qrels`. Se construye `query_lookup` para obtener el texto a partir del identificador de una consulta, y `judgments` para consultar la relevancia anotada de cada documento respecto a esa consulta.

Para la evaluación se utilizan **las 300 consultas que tienen qrels en test**. No se seleccionan únicamente consultas donde un modelo obtenga buenos resultados. Antes de evaluar se comprueba que cada identificador de consulta exista en la colección y que todos los documentos mencionados por los qrels estén presentes en el corpus indexado.

La tabla `queries_used` muestra los identificadores, los textos y el número de documentos relevantes de las consultas utilizadas. Esto deja explícito qué se evaluó y permite repetir los cálculos. Los qrels se reservan para medir la recuperación: no intervienen en la construcción del índice ni en las respuestas que genera el buscador.

In [3]:
from datasets import load_dataset
import pandas as pd
queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_test = load_dataset("BeIR/scifact-qrels", split="test")
query_lookup = {str(row["_id"]): row["text"] for row in queries_ds}
judgments = defaultdict(dict)
for row in qrels_test:
    judgments[str(row["query-id"])][str(row["corpus-id"])] = int(row["score"])
query_ids = sorted(judgments)
assert all(qid in query_lookup for qid in query_ids)
assert all(doc_id in engine.documents for rels in judgments.values() for doc_id in rels)
print("Consultas evaluadas:", len(query_ids), "Relaciones qrels:", len(qrels_test))
queries_used = pd.DataFrame([{"query_id": qid, "query": query_lookup[qid], "relevantes": sum(r>0 for r in judgments[qid].values())} for qid in query_ids])
print(queries_used.to_string(index=False))

Consultas evaluadas: 300 Relaciones qrels: 339
query_id                                                                                                                                                                                                        query  relevantes
       1                                                                                                                                                        0-dimensional biomaterials show inductive properties.           1
     100                                                                                                                                           All hematopoietic stem cells segregate their chromosomes randomly.           1
    1012                                                                                                                               Radioiodine treatment of non-toxic multinodular goitre reduces thyroid volume.           1
    1014                                         

# Fase 4: Evaluación de TF-IDF y BM25

Se ejecuta cada consulta de `test` con los dos modelos y se conservan sus primeros diez resultados en `rankings`. Después, la función `evaluate` compara los identificadores recuperados con los juicios de relevancia de esa consulta. De esta manera se evalúa la recuperación del documento original y no solamente si su texto parece relacionado con la consulta.

`Precision@10` indica cuántos de los diez primeros resultados son relevantes y divide esa cantidad por diez, incluso si se recuperan menos documentos. `Recall@10` indica qué fracción de todos los relevantes conocidos se encontró en esas posiciones. Por ejemplo, recuperar un único relevante en el top 10 puede dar una precisión de `0.1` y un recall de `1.0` cuando ese es el único relevante anotado para la consulta.

`NDCG@10` también toma en cuenta la posición: encontrar un relevante al inicio del ranking aporta más que encontrarlo al final. Se calcula una ganancia `2^rel - 1`, se aplica el descuento `log2(rank + 1)` y se normaliza por el ranking ideal. En los qrels de SciFact la relevancia es binaria.

Las métricas de cada consulta se guardan en `per_query`. Luego se calcula en `aggregate` el promedio de cada métrica sobre todas las consultas evaluadas, incluidas aquellas sin resultados. Se presentan los promedios de ambos modelos, la diferencia `BM25 - TF-IDF` y las métricas por consulta. Una diferencia positiva favorece a BM25 para esa métrica; una negativa favorece a TF-IDF. El archivo local `.data/evaluacion.json` permite conservar esos cálculos sin añadir datos generados a la entrega de Git.

In [4]:
from statistics import fmean

def evaluate(ranking, relevance, k=10):
    ids = [doc_id for doc_id, _ in ranking[:k]]
    assert len(ids) == len(set(ids))
    relevant = {d for d,r in relevance.items() if r>0}
    hits = sum(d in relevant for d in ids)
    dcg = sum((2**max(0,relevance.get(d,0))-1)/math.log2(rank+1) for rank,d in enumerate(ids,1))
    ideal = sorted((max(0,r) for r in relevance.values()),reverse=True)[:k]
    idcg = sum((2**r-1)/math.log2(rank+1) for rank,r in enumerate(ideal,1))
    return {"Precision@10": hits/k, "Recall@10": hits/len(relevant) if relevant else 0.0, "NDCG@10": dcg/idcg if idcg else 0.0}

rankings = {}
per_query = {}
aggregate = {}
for model in ["tfidf", "bm25"]:
    rankings[model] = {qid: engine.rank(query_lookup[qid], model, 10) for qid in query_ids}
    per_query[model] = {qid: evaluate(rankings[model][qid], judgments[qid]) for qid in query_ids}
    aggregate[model] = {m: fmean(row[m] for row in per_query[model].values()) for m in ["Precision@10", "Recall@10", "NDCG@10"]}
print(pd.DataFrame(aggregate).T.to_string())
comparison = pd.DataFrame(aggregate).T
print("Diferencia BM25 - TF-IDF:")
print((comparison.loc["bm25"]-comparison.loc["tfidf"]).to_string())
for model in ["tfidf","bm25"]:
    print("\nMetricas por consulta:",model)
    print(pd.DataFrame.from_dict(per_query[model],orient="index").to_string())
Path(".data/evaluacion.json").write_text(json.dumps({"aggregate":aggregate,"per_query":per_query,"query_ids":query_ids},indent=2),encoding="utf8")

       Precision@10  Recall@10   NDCG@10
tfidf      0.082333   0.748222  0.598481
bm25       0.091000   0.822944  0.690083
Diferencia BM25 - TF-IDF:
Precision@10    0.008667
Recall@10       0.074722
NDCG@10         0.091601

Metricas por consulta: tfidf
      Precision@10  Recall@10   NDCG@10
1              0.0   0.000000  0.000000
100            0.1   1.000000  0.430677
1012           0.1   1.000000  1.000000
1014           0.1   1.000000  1.000000
1019           0.1   1.000000  0.630930
1020           0.1   1.000000  1.000000
1021           0.1   1.000000  1.000000
1024           0.1   1.000000  0.500000
1029           0.2   0.666667  0.671386
1041           0.1   0.500000  0.264068
1049           0.0   0.000000  0.000000
1062           0.1   1.000000  1.000000
1086           0.1   1.000000  1.000000
1088           0.0   0.000000  0.000000
1089           0.1   1.000000  1.000000
1099           0.0   0.000000  0.000000
1100           0.0   0.000000  0.000000
1104           0.1   1.000

67675

# Fase 5: Verificación de los cálculos y del servicio

Se comprueban las métricas con una implementación independiente mediante `ir-measures`. Se utilizan los mismos qrels y los rankings ya generados, con puntuaciones ordinales para mantener exactamente el orden de los resultados, incluso cuando los scores originales empatan. Las aserciones verifican que `Precision@10`, `Recall@10` y `NDCG@10` coincidan dentro de la tolerancia indicada.

También se contrasta el coseno `TF-IDF` propio con `TfidfVectorizer` de scikit-learn usando una muestra de treinta documentos y el mismo preprocesamiento. La biblioteca se utiliza como referencia para comprobar las puntuaciones; el buscador continúa utilizando la implementación propia de `SearchEngine`.

Por último, se prueban los endpoints con `TestClient`. Se verifica que `/health` reconozca el corpus y que `/search` responda con los dos modelos. Las comprobaciones revisan el orden descendente, la ausencia de duplicados, la validez de los identificadores y el respeto de `top_k`. Se incluyen una consulta nueva y un término inexistente para comprobar que la búsqueda no dependa de ejemplos preparados. Las entradas inválidas deben devolver el código HTTP `422`.

In [5]:
import ir_measures
from ir_measures import P, R, nDCG
reference_qrels = [ir_measures.Qrel(qid, d, rel) for qid, judgments_q in judgments.items() for d,rel in judgments_q.items()]
for model in ["tfidf","bm25"]:
    # Puntuaciones ordinales mantienen el desempate del ranking propio.
    run = [ir_measures.ScoredDoc(qid,d,float(11-rank)) for qid,ranking in rankings[model].items() for rank,(d,score) in enumerate(ranking,1)]
    external = ir_measures.calc_aggregate([P@10,R@10,nDCG@10],reference_qrels,run)
    for measure, key in [(P@10,"Precision@10"),(R@10,"Recall@10"),(nDCG@10,"NDCG@10")]:
        assert abs(external[measure]-aggregate[model][key]) < 1e-10
    print(model, "metricas coinciden con ir-measures")

from sklearn.feature_extraction.text import TfidfVectorizer
sample = documents[:30]
small_engine = SearchEngine(sample)
v = TfidfVectorizer(analyzer=preprocess, smooth_idf=True, sublinear_tf=False)
matrix = v.fit_transform([d["title"]+" "+d["text"] for d in sample])
query = sample[0]["title"]
expected = (matrix @ v.transform([query]).T).toarray().ravel()
actual = dict(small_engine.rank(query,"tfidf",len(sample)))
for row, score in zip(sample,expected):
    assert abs(actual.get(str(row["_id"]),0)-score)<1e-10
print("Coseno TF-IDF propio coincide con scikit-learn")

from fastapi.testclient import TestClient
with TestClient(app) as client:
    assert client.get("/health").json()["documents"] == engine.n
    for model in ["tfidf","bm25"]:
        for query in [query_lookup[query_ids[0]],"novel unknown wording about cell membrane signaling", "zzzxqvunknownword"]:
            response = client.post("/search",json={"query":query,"model":model,"top_k":7})
            assert response.status_code==200
            results = response.json()["results"]
            assert len(results)<=7
            assert len({r["document_id"] for r in results})==len(results)
            assert all(r["document_id"] in engine.documents for r in results)
            assert [r["rank"] for r in results]==list(range(1,len(results)+1))
            assert all(results[i]["score"]>=results[i+1]["score"] for i in range(len(results)-1))
        print(model,"API: IDs, orden, top_k y consultas nuevas verificados")
    for body in [{"query":" "},{"query":"x","model":"bge"},{"query":"x","top_k":0},{"query":"x","top_k":True}]:
        assert client.post("/search",json=body).status_code==422
    print("Validaciones HTTP 422 verificadas")
    print(json.dumps(client.post("/search",json={"query":query_lookup[query_ids[0]],"model":"bm25","top_k":3}).json(),indent=2))

from concurrent.futures import ThreadPoolExecutor
parallel_queries = ["Cells activated not inhibited vitamin deficiency DNA repair " * 30] * 16
expected_tokens = [preprocess(q) for q in parallel_queries]
with ThreadPoolExecutor(max_workers=4) as pool:
    assert list(pool.map(preprocess, parallel_queries)) == expected_tokens
print("Preprocesamiento concurrente verificado sin alterar los tokens")


tfidf metricas coinciden con ir-measures
bm25 metricas coinciden con ir-measures


Coseno TF-IDF propio coincide con scikit-learn


tfidf API: IDs, orden, top_k y consultas nuevas verificados
bm25 API: IDs, orden, top_k y consultas nuevas verificados
Validaciones HTTP 422 verificadas
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "10608397",
      "score": 13.109217482568539,
      "title": "High-performance neuroprosthetic control by an individual with tetraplegia.",
      "snippet": "BACKGROUND Paralysis or amputation of an arm results in the loss of the ability to orient the hand and grasp, manipulate, and carry objects, functions that are essential for activities of daily living. Brain-machine interfaces could provide a solution to restoring many of these lost functions. We therefore tested whether an individual with tetraple"
    },
    {
      "rank": 2,
      "document_id": "18953920",
      "score": 9.980722846036928,
      "title": "The Epithelial-Mesenchymal Transition Generates Cells with Properties of Stem Cells",
      "snippet": "The epithelial-mesenchymal transition (

Preprocesamiento concurrente verificado sin alterar los tokens


# Fase 6: Comparación de los rankings e interpretación de los resultados

Se comparan las métricas agregadas y se calcula, para cada consulta, la diferencia de `NDCG@10` entre BM25 y TF-IDF. A partir de esas diferencias se selecciona un caso favorable a cada modelo. Esta selección sirve para examinar errores y aciertos concretos, después de haber evaluado el conjunto completo.

Para cada ejemplo se muestra la consulta, sus métricas y los tres primeros documentos recuperados, junto con su título y su condición de relevancia anotada. Esto permite observar si un modelo coloca un relevante en las primeras posiciones o si lo deja fuera del top 10. Las posiciones y las métricas se comparan entre modelos; las magnitudes de sus scores pertenecen a escalas diferentes.

Los ejemplos se interpretan como resultados de esta ejecución sobre SciFact. No se utilizan para cambiar los parámetros después de observar `test`, ni se asume que el modelo con mejor promedio vaya a resolver mejor todas las consultas.

In [6]:
deltas = {qid: per_query["bm25"][qid]["NDCG@10"]-per_query["tfidf"][qid]["NDCG@10"] for qid in query_ids}
for model in ["tfidf","bm25"]:
    print(model, ":", aggregate[model])
for name, reverse in [("BM25 mejora TF-IDF",True),("TF-IDF mejora BM25",False)]:
    qid = sorted(deltas,key=lambda q:deltas[q],reverse=reverse)[0]
    print("\n",name,"consulta",qid,query_lookup[qid],"delta NDCG",deltas[qid])
    for model in ["tfidf","bm25"]:
        print(model,per_query[model][qid])
        for doc_id,score in rankings[model][qid][:3]:
            print(doc_id,round(score,4),"RELEVANTE" if judgments[qid].get(doc_id,0)>0 else "NO ANOTADO",engine.documents[doc_id]["title"])
winner = max(aggregate,key=lambda m:aggregate[m]["NDCG@10"])
print("\nMayor NDCG@10 promedio en esta ejecucion:",winner)
print("BM25 satura TF y ajusta longitud; TF-IDF usa TF lineal y normalizacion coseno. Por eso una repeticion frecuente o un documento largo pueden cambiar sus posiciones de manera distinta. La comparacion de ejemplos debe atender a la relevancia anotada, no a la magnitud de scores entre modelos.")
print("Limitacion: los documentos sin anotacion se consideran no relevantes para la metrica; no implica que sean cientificamente irrelevantes.")

tfidf : {'Precision@10': 0.08233333333333334, 'Recall@10': 0.7482222222222222, 'NDCG@10': 0.598481445665445}
bm25 : {'Precision@10': 0.091, 'Recall@10': 0.8229444444444444, 'NDCG@10': 0.6900825720536997}

 BM25 mejora TF-IDF consulta 1137 TNFAIP3 is a tumor suppressor in glioblastoma. delta NDCG 1.0
tfidf {'Precision@10': 0.0, 'Recall@10': 0.0, 'NDCG@10': 0.0}
18600579 0.2354 NO ANOTADO Development of a novel mouse glioma model using lentiviral vectors
22901758 0.2298 NO ANOTADO Hedgehog signaling regulates brain tumor-initiating cell proliferation and portends shorter survival for patients with PTEN-coexpressing glioblastomas.
11578459 0.2279 NO ANOTADO Chromosome 7 gain and DNA hypermethylation at the HOXA10 locus are associated with expression of a stem cell related HOX-signature in glioblastoma
bm25 {'Precision@10': 0.1, 'Recall@10': 1.0, 'NDCG@10': 1.0}
33370 21.2174 RELEVANTE Targeting A20 Decreases Glioma Stem Cell Survival and Tumor Growth
18600579 16.8531 NO ANOTADO Developmen

## Interpretación de los resultados obtenidos

En esta ejecución, **BM25 obtiene mejores promedios en las tres métricas**. Su `Precision@10` es `0.0910`, frente a `0.0823` de TF-IDF; su `Recall@10` es `0.8229`, frente a `0.7482`; y su `NDCG@10` es `0.6901`, frente a `0.5985`. La mejora absoluta del recall es de aproximadamente **7,47 puntos porcentuales**, mientras que la de NDCG es de `0.0916`. En conjunto, BM25 recupera una mayor proporción de relevantes conocidos y obtiene una mejor colocación de esos documentos en el ranking.

La consulta `1137`, sobre TNFAIP3 como supresor tumoral, muestra una ventaja concreta de BM25. Este modelo sitúa el documento relevante `33370` en primer lugar y alcanza `NDCG@10 = 1`. En cambio, TF-IDF no lo recupera entre sus diez primeros resultados y obtiene `NDCG@10 = 0`. En ese caso, la diferencia no es solamente de posición: el documento relevante queda fuera del corte evaluado para TF-IDF.

En la consulta `820`, sobre *N-terminal cleavage*, ocurre lo contrario. TF-IDF coloca primero el documento relevante `8646760` y alcanza `NDCG@10 = 1`, mientras BM25 no lo incluye en el top 10. Este ejemplo muestra por qué el mejor promedio de BM25 no implica que sea superior en todas las consultas.

La precisión parece baja si se observa de forma aislada, pero debe interpretarse junto con la cantidad de relevantes anotados. Cuando una consulta tiene pocos relevantes, incluso recuperarlos todos deja un valor bajo de `Precision@10`, porque su denominador sigue siendo diez. Por eso se analizan conjuntamente precisión, recall y posición de los relevantes, en lugar de concluir que el sistema falla únicamente por el valor de precisión.

Una diferencia entre los modelos es que BM25 satura el aporte de las repeticiones y ajusta por longitud, mientras TF-IDF utiliza frecuencia lineal y normalización coseno. Esas decisiones pueden producir rankings diferentes, pero los ejemplos mostrados no permiten atribuir cada resultado a una única causa sin realizar experimentos adicionales. Además, un documento sin anotación se considera no relevante para estas métricas; eso no demuestra que su contenido sea científicamente irrelevante para la consulta.

# Fase 7: Publicación y comprobación de la API en internet

Se publica el servicio en **https://exb1-martincusme-scifact.vercel.app** para que pueda consultarse desde un cliente como Postman o desde la documentación interactiva en `/docs`. El endpoint `GET /health` comprueba que el corpus puede cargarse, mientras `POST /search` recibe `query`, `model` y `top_k`. El campo `model` admite `bm25` y `tfidf`.

Para realizar una solicitud se envía, por ejemplo, `{"query":"Vitamin D deficiency is associated with cardiovascular disease.","model":"bm25","top_k":10}` como cuerpo JSON a `/search`. La respuesta contiene el modelo utilizado y una lista de documentos con `rank`, `document_id`, `score`, `title` y `snippet`. El snippet recorta únicamente la presentación del texto; el documento se indexa completo. La misma solicitud puede repetirse cambiando el modelo a `tfidf`.

El bloque siguiente prueba la URL pública y compara sus resultados con los del motor local. Se comprueban los identificadores, el formato, el orden, la cantidad solicitada y la respuesta ante entradas inválidas o sin coincidencias. Así se verifica el servicio realmente publicado, además de las pruebas locales. Las consultas se procesan con el índice: no se responden mediante qrels ni con rankings guardados de antemano.

Para reproducir la publicación se ejecuta `vercel deploy --prod --yes` desde esta carpeta. La entrega conserva el notebook ejecutado y los archivos auxiliares necesarios para alojar el mismo código. Los datos descargados, los entornos y las credenciales quedan excluidos de Git.

In [7]:
import httpx
PUBLIC_URL = "https://exb1-martincusme-scifact.vercel.app"
with httpx.Client(base_url=PUBLIC_URL, timeout=240) as remote:
    response = remote.get("/health")
    response.raise_for_status()
    assert response.json()["documents"]==5183
    print("GET /health",response.status_code,response.json())
    for model in ["tfidf","bm25"]:
        # Esta frase libre no se usa para construir el indice ni contiene qrels.
        query = "Vitamin D deficiency and cardiovascular disease risk"
        response = remote.post("/search",json={"query":query,"model":model,"top_k":5})
        response.raise_for_status()
        data = response.json()
        assert data["model"]==model
        results=data["results"]
        assert len(results)==5
        assert [r["document_id"] for r in results]==[d for d,s in engine.rank(query,model,5)]
        assert all(r["document_id"] in engine.documents for r in results)
        assert len({r["document_id"] for r in results})==5
        assert all(set(["rank","document_id","score","title","snippet"])<=set(r) for r in results)
        assert all(results[i]["score"]>=results[i+1]["score"] for i in range(4))
        print("POST /search",model,response.status_code,"IDs originales y ranking coinciden con ejecucion local")
        print(json.dumps(data,indent=2))
    assert remote.post("/search",json={"query":" ","model":"bm25"}).status_code==422
    assert remote.post("/search",json={"query":"zzzxqvunknownword","model":"tfidf","top_k":10}).json()["results"]==[]
    print("Validacion y consulta sin coincidencias verificadas en la URL publica")

GET /health 200 {'status': 'ok', 'corpus': 'SciFact', 'documents': 5183, 'models': ['tfidf', 'bm25']}


POST /search tfidf 200 IDs originales y ranking coinciden con ejecucion local
{
  "model": "tfidf",
  "results": [
    {
      "rank": 1,
      "document_id": "23267371",
      "score": 0.6229171489476348,
      "title": "Vitamin D: The \"sunshine\" vitamin.",
      "snippet": "Vitamin D insufficiency affects almost 50% of the population worldwide. An estimated 1 billion people worldwide, across all ethnicities and age groups, have a vitamin D deficiency (VDD). This pandemic of hypovitaminosis D can mainly be attributed to lifestyle (for example, reduced outdoor activities) and environmental (for example, air pollution) f"
    },
    {
      "rank": 2,
      "document_id": "30720103",
      "score": 0.5491620811725672,
      "title": "Vitamin D status: measurement, interpretation, and clinical application.",
      "snippet": "Vitamin D, the sunshine vitamin, is now recognized not only for its importance in promoting bone health in children and adults but also for other health benefits,

POST /search bm25 200 IDs originales y ranking coinciden con ejecucion local
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "30720103",
      "score": 26.936973652085584,
      "title": "Vitamin D status: measurement, interpretation, and clinical application.",
      "snippet": "Vitamin D, the sunshine vitamin, is now recognized not only for its importance in promoting bone health in children and adults but also for other health benefits, including reducing the risk of chronic diseases such as autoimmune diseases, common cancer, and cardiovascular disease. Vitamin D made in the skin or ingested in the diet is biologically "
    },
    {
      "rank": 2,
      "document_id": "23267371",
      "score": 24.51300311593551,
      "title": "Vitamin D: The \"sunshine\" vitamin.",
      "snippet": "Vitamin D insufficiency affects almost 50% of the population worldwide. An estimated 1 billion people worldwide, across all ethnicities and age groups, have a vitami

Validacion y consulta sin coincidencias verificadas en la URL publica
